# basic Cleaning and auditing of the dataset

In [101]:
import pandas as pd

starting with dataset

In [102]:
df = pd.read_csv("../data/processed/online_retail_combined.csv", parse_dates=["InvoiceDate"])
print(df.shape)
df.head()

(1067371, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [103]:
df.columns
df['Revenue'] = df['Quantity'].abs().multiply(df['UnitPrice'])
print(df.describe()) #summary statistics for numeric columns
print(df.columns)    #Show columns headings
print(df.info())    #Show data frame into summary


           Quantity                 InvoiceDate     UnitPrice     CustomerID  \
count  1.067371e+06                     1067371  1.067371e+06  824364.000000   
mean   9.938898e+00  2011-01-02 21:13:55.394029  4.649388e+00   15324.638504   
min   -8.099500e+04         2009-12-01 07:45:00 -5.359436e+04   12346.000000   
25%    1.000000e+00         2010-07-09 09:46:00  1.250000e+00   13975.000000   
50%    3.000000e+00         2010-12-07 15:28:00  2.100000e+00   15255.000000   
75%    1.000000e+01         2011-07-22 10:23:00  4.150000e+00   16797.000000   
max    8.099500e+04         2011-12-09 12:50:00  3.897000e+04   18287.000000   
std    1.727058e+02                         NaN  1.235531e+02    1697.464450   

            Revenue  
count  1.067371e+06  
mean   2.093118e+01  
min   -5.359436e+04  
25%    3.750000e+00  
50%    9.950000e+00  
75%    1.770000e+01  
max    1.684696e+05  
std    2.922293e+02  
Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
      

In [104]:
print("\n",df.isna().sum())  #null values


 InvoiceNo           0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     243007
Country             0
Revenue             0
dtype: int64


Missing Values and duplicates

In [105]:
print("missing Customer id's :" , df['CustomerID'].isnull().sum())
print("Quantity of zero or below :" , (df['Quantity'] <= 0).sum())
print("Cancelled invoices :" , (df["InvoiceNo"].str.startswith(('c','C'))).sum())
print("Price of zero or below :" , (df['UnitPrice'] <= 0).sum())
print("Non-product stock codes :" , (~df['StockCode'].str.match(r"^\d{5}")).sum())
print("Missing descriptions :" , df['Description'].isnull().sum())
print("Exact duplicate rows :" , df.duplicated().sum())

missing Customer id's : 243007
Quantity of zero or below : 22950
Cancelled invoices : 19494
Price of zero or below : 6207
Non-product stock codes : 6093
Missing descriptions : 4382
Exact duplicate rows : 34335


Handling canceled invoices

In [ ]:
# separating cancelled invoices
df = df.drop_duplicates()

is_cancelled = df["InvoiceNo"].str.startswith(("C", "c"))
returns_raw = df[is_cancelled].copy()
returns_raw["ReturnValue"] = (returns_raw["Quantity"].abs() * returns_raw["UnitPrice"]).round(2)
returns_raw = returns_raw.drop(columns=["Revenue"], errors="ignore")
returns_raw = returns_raw.dropna(subset=["CustomerID"])
sales_raw = df[~is_cancelled].copy()


print(df.shape, returns_raw.shape, sales_raw.shape)


(1033036, 9) (18390, 9) (1013932, 9)


In [107]:
#non stock codes removed 
sales_raw = sales_raw[sales_raw['StockCode'].str.match(r"^\d{5}")]

# Negative Quantity removed 
sales_raw = sales_raw[sales_raw['UnitPrice'] > 0]
clean = sales_raw[~sales_raw['CustomerID'].isnull()]

print(sales_raw.shape , df.shape)

(1003214, 9) (1033036, 9)


Checks

In [108]:
checks = {
    "No missing Customer ID":        clean["CustomerID"].isnull().sum() == 0,
    "No missing Invoice":            clean["InvoiceNo"].isnull().sum() == 0,
    "All quantities > 0":            (clean["Quantity"] > 0).all(),
    "All prices > 0":                (clean["UnitPrice"] > 0).all(),
    "No cancelled invoices in sales": not clean["InvoiceNo"].str.startswith(("C", "c")).any(),
    "No duplicate rows":             clean.duplicated().sum() == 0,
    "Date range looks right":        clean["InvoiceDate"].min().year == 2009 and clean["InvoiceDate"].max().year == 2011,
}
for name, ok in checks.items():
    print("PASS" if ok else "FAIL", "-", name)

print("\nDate range:", clean["InvoiceDate"].min(), "to", clean["InvoiceDate"].max())
print("Customers:", clean["CustomerID"].nunique())
print("Countries:", clean["Country"].nunique())
print("Total revenue:", round(clean["Revenue"].sum(), 2))

PASS - No missing Customer ID
PASS - No missing Invoice
PASS - All quantities > 0
PASS - All prices > 0
PASS - No cancelled invoices in sales
PASS - No duplicate rows
PASS - Date range looks right

Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00
Customers: 5852
Countries: 41
Total revenue: 17068567.97


Save the output

In [109]:
import os
os.makedirs("../data/processed", exist_ok=True)

clean.to_csv("../data/processed/sales_clean.csv", index=False)
returns_raw.to_csv("../data/processed/returns_clean.csv", index=False)

# Small sample 
clean.head(1000).to_csv("../data/sample_online_retail.csv", index=False)
